# **MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**

# Análisis de Grandes Volúmenes de datos

## Instituto Tecnológico de Monterrey

## Actividad 3: Aprendizaje Supervisado y No Supervisado con PySpark

---

Alumno : Emanuel Flores Martínez

Matrícula: A01796497

---

Este notebook da continuidad directa a la **Evidencia 1** del equipo, donde se
caracterizó la población *P* (pares **BTCUSDT, ETHUSDT, XRPUSDT**), se definió el
**particionamiento** `A = symbol × B = volatility_level × C = market_period` y se
propuso el **muestreo aleatorio estratificado** (`sampleBy`) para construir la
muestra *M*. Aquí reutilizamos ese mismo diseño y aplicamos sobre *M* un algoritmo
de aprendizaje **supervisado** y uno **no supervisado** implementados en PySpark MLlib.

# 1. Introducción teórica

## 1.1 Aprendizaje supervisado
El **aprendizaje supervisado** entrena un modelo a partir de ejemplos *etiquetados*:
cada instancia incluye un vector de características `X` y una variable objetivo `y`
conocida. El objetivo es aprender una función `f(X) ≈ y` que generalice a datos
nuevos. Se divide en:

- **Clasificación**: la variable objetivo es categórica (p. ej. *bullish / bearish /
  neutral*). Algoritmos representativos en la literatura: **árboles de decisión**,
  **Random Forest**, **Gradient-Boosted Trees**, **regresión logística**, **SVM**,
  **Naive Bayes**, **redes neuronales (Multilayer Perceptron)**.
- **Regresión**: la variable objetivo es numérica continua (p. ej. el precio futuro).
  Algoritmos: **regresión lineal**, **árboles/bosques de regresión**, **GBT regresor**.

Métricas típicas de calidad: *accuracy*, *precision*, *recall*, *F1*, matriz de
confusión, AUC (clasificación); RMSE, MAE, R² (regresión).

## 1.2 Aprendizaje no supervisado
El **aprendizaje no supervisado** trabaja con datos **sin etiquetar**: busca
estructura intrínseca en los datos. Las tareas principales son:

- **Agrupamiento (clustering)**: descubrir grupos de instancias similares.
  Algoritmos representativos: **K-Means**, **Gaussian Mixture Models (GMM)**,
  **Power Iteration Clustering (PIC)**, **Bisecting K-Means**, **DBSCAN**,
  **clustering jerárquico**.
- **Reducción de dimensionalidad**: **PCA**, **SVD**.
- **Reglas de asociación**: **FP-Growth**.

Como no hay etiqueta de referencia, la calidad se mide con criterios internos:
**coeficiente de silueta (silhouette)**, inercia (WSSSE / *within-cluster sum of
squares*), índice de Davies-Bouldin, etc.

## 1.3 Algoritmos disponibles en PySpark (`pyspark.ml`)
Spark MLlib (API `pyspark.ml`, basada en DataFrames) ofrece, entre otros:

| Tipo | Algoritmos disponibles en PySpark |
|------|-----------------------------------|
| **Supervisado — Clasificación** | `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`, `GBTClassifier`, `MultilayerPerceptronClassifier`, `LinearSVC`, `NaiveBayes`, `FMClassifier` |
| **Supervisado — Regresión** | `LinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor`, `GBTRegressor`, `GeneralizedLinearRegression` |
| **No supervisado — Clustering** | `KMeans`, `BisectingKMeans`, `GaussianMixture`, `PowerIterationClustering (PIC)`, `LDA` |
| **No supervisado — Reducción dim.** | `PCA`, `SVD` |
| **Soporte / *feature engineering*** | `VectorAssembler`, `StringIndexer`, `OneHotEncoder`, `StandardScaler`, `Pipeline`, `CrossValidator` |

**Selección para esta actividad:**
- **Supervisado → `RandomForestClassifier`**: robusto a outliers, no requiere
  escalado, maneja relaciones no lineales y entrega *importancia de variables*
  (interpretabilidad). Variable objetivo: `market_direction`.
- **No supervisado → `KMeans`**: eficiente y escalable en Spark, fácil de
  interpretar mediante los centroides; la calidad se evalúa con *silhouette*.

# 2. Selección de los datos

Reutilizamos la conexión a Drive y el dataset *Binance Full History* de la
Evidencia 1. Para mantener **tiempos de procesamiento contenidos** trabajamos solo
con los **tres pares de la población** (BTC/ETH/XRP) en lugar de los ~1000 pares del
dataset completo, y luego construimos la muestra *M* por **muestreo estratificado**.


In [1]:
import os, zipfile, fnmatch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Kaggle credentials
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
!pip install -q kaggle

DATASET     = "jorijnsmit/binance-full-history"
WANTED      = ["BTC", "ETH", "XRP"]                                       # pairs vs USDT

EXTRACT_DIR = Path("/content/drive/MyDrive/binance_full_history/files")   # only 3 parquet, on Drive
TMP_DIR     = Path("/content/_binance_tmp")                               # ephemeral ZIP
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# 1. If the 3 parquet files already exist on Drive -> nothing to do
existing = sorted(EXTRACT_DIR.glob("*.parquet"))
def _norm(s):
    return ''.join(c for c in s.upper() if c.isalnum())
already = {b for b in WANTED if any(b in _norm(p.stem) and "USDT" in _norm(p.stem) for p in existing)}

if already == set(WANTED):
    print("The 3 parquet files are already on Drive - skipping download.")
    print("Files:", [p.name for p in existing])
else:
    missing = set(WANTED) - already
    print(f"Missing on Drive: {missing}. Downloading ZIP to ephemeral disk...")

    # 2. Download ZIP to /content (ephemeral, ~26 GB, fits on Colab's disk)
    zip_files = list(TMP_DIR.glob("*.zip"))
    if not zip_files:
        get_ipython().system(f'kaggle datasets download -d {DATASET} -p {TMP_DIR}')
        zip_files = list(TMP_DIR.glob("*.zip"))
    ZIP_PATH = zip_files[0]
    print(f"ZIP: {ZIP_PATH}  ({ZIP_PATH.stat().st_size / (1024**3):.2f} GB)")

    # 3. Extract ONLY the 3 pairs straight to Drive
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        names = z.namelist()
        for base in WANTED:
            matches = [n for n in names if fnmatch.fnmatch(n.upper(), f"*{base}*USDT*.PARQUET")]
            for n in matches:
                dest = EXTRACT_DIR / Path(n).name      # flatten path -> straight into EXTRACT_DIR
                with z.open(n) as src, open(dest, "wb") as out:
                    out.write(src.read())
                print("Extracted to Drive:", dest.name)

    # 4. Delete the ephemeral ZIP to free space
    ZIP_PATH.unlink()
    print("Ephemeral ZIP deleted.")

print("\nParquet files available on Drive:", sorted(p.name for p in EXTRACT_DIR.glob("*.parquet")))

Mounted at /content/drive
The 3 parquet files are already on Drive - skipping download.
Files: ['BTC-USDT.parquet', 'BTCDOWN-USDT.parquet', 'BTCST-USDT.parquet', 'BTCUP-USDT.parquet', 'ETH-USDT.parquet', 'ETHDOWN-USDT.parquet', 'ETHUP-USDT.parquet', 'XRP-USDT.parquet', 'XRPDOWN-USDT.parquet', 'XRPUP-USDT.parquet']

Parquet files available on Drive: ['BTC-USDT.parquet', 'BTCDOWN-USDT.parquet', 'BTCST-USDT.parquet', 'BTCUP-USDT.parquet', 'ETH-USDT.parquet', 'ETHDOWN-USDT.parquet', 'ETHUP-USDT.parquet', 'XRP-USDT.parquet', 'XRPDOWN-USDT.parquet', 'XRPUP-USDT.parquet']


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Binance ML Supervised and Unsupervised")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.files.maxPartitionBytes", "128MB")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.0.2


In [3]:
# Robust lookup of each pair's parquet file.
# The dataset names files per pair; depending on the source it can be
# 'BTCUSDT.parquet' or 'BTC-USDT.parquet'. We normalize names to match them.
import glob

SYMBOLS = ["BTCUSDT", "ETHUSDT", "XRPUSDT"]

all_files = glob.glob(os.path.join(str(EXTRACT_DIR), "*.parquet"))
print(f"Total parquet files available: {len(all_files)}")

def find_pair_file(symbol):
    target = _norm(symbol)
    for f in all_files:
        if _norm(os.path.basename(f).replace('.parquet', '')) == target:
            return f
    return None

pair_files = {sym: find_pair_file(sym) for sym in SYMBOLS}
for sym, f in pair_files.items():
    print(f"{sym:>8} -> {f}")

missing_files = [s for s, f in pair_files.items() if f is None]
assert not missing_files, (
    f"No parquet found for: {missing_files}. "
    f"Check the real names with: print(sorted(os.path.basename(x) for x in all_files)[:20])"
)

Total parquet files available: 10
 BTCUSDT -> /content/drive/MyDrive/binance_full_history/files/BTC-USDT.parquet
 ETHUSDT -> /content/drive/MyDrive/binance_full_history/files/ETH-USDT.parquet
 XRPUSDT -> /content/drive/MyDrive/binance_full_history/files/XRP-USDT.parquet


In [4]:
# Load the 3 pairs and union them, adding a 'symbol' column
from functools import reduce
from pyspark.sql.functions import lit

dfs = []
for sym in SYMBOLS:
    d = spark.read.parquet(pair_files[sym]).withColumn("symbol", lit(sym))
    dfs.append(d)

df3 = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

df3.printSchema()
print("Columns:", df3.columns)

root
 |-- open: float (nullable = true)
 |-- high: float (nullable = true)
 |-- low: float (nullable = true)
 |-- close: float (nullable = true)
 |-- volume: float (nullable = true)
 |-- quote_asset_volume: float (nullable = true)
 |-- number_of_trades: integer (nullable = true)
 |-- taker_buy_base_asset_volume: float (nullable = true)
 |-- taker_buy_quote_asset_volume: float (nullable = true)
 |-- open_time: timestamp_ntz (nullable = true)
 |-- symbol: string (nullable = false)

Columns: ['open', 'high', 'low', 'close', 'volume', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'open_time', 'symbol']


In [5]:
# Row count per pair (load check)
from pyspark.sql.functions import count
df3.groupBy("symbol").agg(count("*").alias("rows")).show()

+-------+-------+
| symbol|   rows|
+-------+-------+
|BTCUSDT|2753182|
|ETHUSDT|2753105|
|XRPUSDT|2381218|
+-------+-------+



## 2.2 Variables derivadas de caracterización

Recreamos las variables derivadas definidas en la Evidencia 1, que constituyen la
base tanto del particionamiento como de los modelos:

- `return_pct = (close - open) / open * 100`
- `intrabar_volatility_pct = (high - low) / open * 100`
- `taker_buy_ratio = taker_buy_base_asset_volume / volume` (presión compradora)
- `market_direction` ∈ {bullish, bearish, neutral} — **variable objetivo supervisada**
- `market_session` ∈ {Asia, Europe, America}
- `hour` (hora UTC de la vela)

In [6]:
from pyspark.sql.functions import col, when, hour

df_feat = (
    df3
    # keep only candles with a valid opening price (avoids invalid divisions)
    .filter(col("open").isNotNull() & (col("open") > 0))
    .withColumn("return_pct", (col("close") - col("open")) / col("open") * 100)
    .withColumn("intrabar_volatility_pct", (col("high") - col("low")) / col("open") * 100)
    .withColumn(
        "taker_buy_ratio",
        when(col("volume") > 0, col("taker_buy_base_asset_volume") / col("volume")).otherwise(0.0),
    )
    .withColumn("hour", hour(col("open_time")))
    .withColumn(
        "market_direction",
        when(col("close") > col("open"), "bullish")
        .when(col("close") < col("open"), "bearish")
        .otherwise("neutral"),
    )
    .withColumn(
        "market_session",
        when(col("hour") < 8, "Asia")
        .when(col("hour") < 16, "Europe")
        .otherwise("America"),
    )
)

df_feat.select(
    "symbol", "open_time", "open", "close", "return_pct",
    "intrabar_volatility_pct", "taker_buy_ratio", "market_direction", "market_session"
).show(5, truncate=False)

+-------+-------------------+-------+-------+----------+-----------------------+-------------------+----------------+--------------+
|symbol |open_time          |open   |close  |return_pct|intrabar_volatility_pct|taker_buy_ratio    |market_direction|market_session|
+-------+-------------------+-------+-------+----------+-----------------------+-------------------+----------------+--------------+
|BTCUSDT|2017-08-17 04:00:00|4261.48|4261.48|0.0       |0.0                    |0.04235225229284669|neutral         |Asia          |
|BTCUSDT|2017-08-17 04:01:00|4261.48|4261.48|0.0       |0.0                    |0.0                |neutral         |Asia          |
|BTCUSDT|2017-08-17 04:02:00|4280.56|4280.56|0.0       |0.0                    |1.0                |neutral         |Asia          |
|BTCUSDT|2017-08-17 04:03:00|4261.48|4261.48|0.0       |0.0                    |1.0                |neutral         |Asia          |
|BTCUSDT|2017-08-17 04:04:00|4261.48|4261.48|0.0       |0.0          

## 2.3 Variable de partición `volatility_level` (dimensión B)

Tal como en la Evidencia 1, el nivel de volatilidad se define **por símbolo**
usando percentiles de `intrabar_volatility_pct`:

- **LOW**  ≤ percentil 89
- **MEDIUM** entre p89 y p99
- **HIGH** > percentil 99

Esto reproduce la distribución reportada (LOW ≈ 89 %, MEDIUM ≈ 10 %, HIGH ≈ 1 %)
y respeta que cada activo tiene una escala de volatilidad distinta.

In [7]:
# Per-symbol volatility thresholds (approxQuantile).
# relativeError=0.001 (not 0.01): with heavy-tailed data a coarse error makes
# the p99 estimate collapse onto the maximum, leaving the HIGH stratum empty.
thresholds = {}
for sym in SYMBOLS:
    q = (
        df_feat.filter(col("symbol") == sym)
        .approxQuantile("intrabar_volatility_pct", [0.89, 0.99], 0.001)
    )
    thresholds[sym] = q
    print(f"{sym}: p89={q[0]:.4f}%  p99={q[1]:.4f}%")

# Small DataFrame with thresholds to join (avoids UDFs)
thr_rows = [(sym, float(thresholds[sym][0]), float(thresholds[sym][1])) for sym in SYMBOLS]
thr_df = spark.createDataFrame(thr_rows, ["symbol", "p89", "p99"])

BTCUSDT: p89=0.2630%  p99=0.7346%
ETHUSDT: p89=0.3188%  p99=0.8876%
XRPUSDT: p89=0.3380%  p99=0.9377%


In [8]:
# Assign volatility_level and market_period
from pyspark.sql.functions import year

df_lvl = (
    df_feat.join(thr_df, on="symbol", how="left")
    .withColumn(
        "volatility_level",
        when(col("intrabar_volatility_pct") <= col("p89"), "LOW")
        .when(col("intrabar_volatility_pct") <= col("p99"), "MEDIUM")
        .otherwise("HIGH"),
    )
    # market_period (dimension C) based on the crypto ecosystem's historical ranges
    .withColumn(
        "market_period",
        when(year(col("open_time")) < 2020, "EARLY")
        .when(year(col("open_time")) <= 2021, "BULL")
        .otherwise("MODERN"),
    )
    .drop("p89", "p99")
)

print("=== Distribution by volatility_level ===")
df_lvl.groupBy("symbol", "volatility_level").agg(count("*").alias("n")).orderBy("symbol", "volatility_level").show()
print("=== Distribution by market_period ===")
df_lvl.groupBy("market_period").agg(count("*").alias("n")).show()

=== Distribution by volatility_level ===
+-------+----------------+-------+
| symbol|volatility_level|      n|
+-------+----------------+-------+
|BTCUSDT|            HIGH|  30023|
|BTCUSDT|             LOW|2448400|
|BTCUSDT|          MEDIUM| 274759|
|ETHUSDT|            HIGH|  28625|
|ETHUSDT|             LOW|2447703|
|ETHUSDT|          MEDIUM| 276777|
|XRPUSDT|            HIGH|  25868|
|XRPUSDT|             LOW|2119326|
|XRPUSDT|          MEDIUM| 236024|
+-------+----------------+-------+

=== Distribution by market_period ===
+-------------+-------+
|market_period|      n|
+-------------+-------+
|       MODERN|1382322|
|         BULL|3151164|
|        EARLY|3354019|
+-------------+-------+



## 2.4 Construcción de la muestra *M* por muestreo estratificado

Aplicamos la técnica propuesta en la Evidencia 1: **muestreo aleatorio estratificado**
con `DataFrame.sampleBy()`. El estrato combina las tres dimensiones de
particionamiento (`symbol × volatility_level × market_period`), de modo que cada
partición de las 27 posibles es una unidad de muestreo independiente.

Usamos una **fracción proporcional uniforme** entre estratos: así *M* conserva la
distribución conjunta de la población *P* (incluida la rareza de los regímenes HIGH),
**minimizando el sesgo de selección**. La fracción se calibra para obtener una
muestra de dimensión contenida (objetivo ≈ 60–80 mil instancias).


In [9]:
# Stratum key = partition (A x B x C)
from pyspark.sql.functions import concat_ws

df_strat = df_lvl.withColumn(
    "stratum", concat_ws("|", col("symbol"), col("volatility_level"), col("market_period"))
).cache()

total_rows = df_strat.count()
print(f"Total rows (3 pairs): {total_rows:,}")

# Proportional fraction for a contained sample (~70k)
TARGET_M = 70_000
FRACTION = min(1.0, TARGET_M / total_rows)
print(f"Proportional sampling fraction: {FRACTION:.5f}")

strata = [r["stratum"] for r in df_strat.select("stratum").distinct().collect()]
fractions = {s: FRACTION for s in strata}
print(f"Number of strata (present partitions): {len(strata)}")

Total rows (3 pairs): 7,887,505
Proportional sampling fraction: 0.00887
Number of strata (present partitions): 27


In [10]:
# Stratified sampling: build M
M = df_strat.sampleBy("stratum", fractions=fractions, seed=42).cache()

n_M = M.count()
print(f"Sample M size: {n_M:,} instances")

print("\n=== M composition by stratum (representativeness check) ===")
M.groupBy("symbol", "volatility_level").agg(count("*").alias("n")).orderBy("symbol", "volatility_level").show(30)

Sample M size: 70,304 instances

=== M composition by stratum (representativeness check) ===
+-------+----------------+-----+
| symbol|volatility_level|    n|
+-------+----------------+-----+
|BTCUSDT|            HIGH|  277|
|BTCUSDT|             LOW|21988|
|BTCUSDT|          MEDIUM| 2350|
|ETHUSDT|            HIGH|  277|
|ETHUSDT|             LOW|21844|
|ETHUSDT|          MEDIUM| 2479|
|XRPUSDT|            HIGH|  227|
|XRPUSDT|             LOW|18703|
|XRPUSDT|          MEDIUM| 2159|
+-------+----------------+-----+



# 3. Preparación de los datos

Sobre la muestra *M* aplicamos estrategias de corrección para dejarla lista para los
algoritmos:

1. **Valores nulos**: conteo y eliminación/sustitución según corresponda.
2. **Valores atípicos (outliers)**: detección por IQR y **winsorización** (recorte a
   p1/p99) de variables muy sesgadas (`volume`, `quote_asset_volume`,
   `number_of_trades`) para que no distorsionen al clustering.
3. **Transformación de tipos**: aseguramos tipos numéricos (`double`) para las
   variables que alimentarán a los modelos.


In [11]:
# 3.1 Null-value diagnosis in M
from pyspark.sql.functions import isnan

cols_check = ["open", "high", "low", "close", "volume", "quote_asset_volume",
              "number_of_trades", "taker_buy_base_asset_volume",
              "taker_buy_quote_asset_volume", "return_pct",
              "intrabar_volatility_pct", "taker_buy_ratio"]

null_counts = M.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) for c in cols_check
])
print("=== Null / NaN values per column (in M) ===")
null_counts.show(truncate=False, vertical=True)

=== Null / NaN values per column (in M) ===
-RECORD 0---------------------------
 open                         | 0   
 high                         | 0   
 low                          | 0   
 close                        | 0   
 volume                       | 0   
 quote_asset_volume           | 0   
 number_of_trades             | 0   
 taker_buy_base_asset_volume  | 0   
 taker_buy_quote_asset_volume | 0   
 return_pct                   | 0   
 intrabar_volatility_pct      | 0   
 taker_buy_ratio              | 0   



In [12]:
# 3.2 Null cleanup and type casting
from pyspark.sql.types import DoubleType

num_cols = ["volume", "quote_asset_volume", "number_of_trades",
            "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume",
            "return_pct", "intrabar_volatility_pct", "taker_buy_ratio"]

M_clean = M
# Drop rows missing the minimum information required by the models
M_clean = M_clean.dropna(subset=["close", "open", "volume", "number_of_trades"])
# Force numeric double type (number_of_trades usually comes as long)
for c in num_cols:
    M_clean = M_clean.withColumn(c, col(c).cast(DoubleType()))

print(f"Rows after dropping critical nulls: {M_clean.count():,}")
M_clean.select(num_cols).describe().show()

Rows after dropping critical nulls: 70,304
+-------+-----------------+------------------+------------------+---------------------------+----------------------------+--------------------+-----------------------+-------------------+
|summary|           volume|quote_asset_volume|  number_of_trades|taker_buy_base_asset_volume|taker_buy_quote_asset_volume|          return_pct|intrabar_volatility_pct|    taker_buy_ratio|
+-------+-----------------+------------------+------------------+---------------------------+----------------------------+--------------------+-----------------------+-------------------+
|  count|            70304|             70304|             70304|                      70304|                       70304|               70304|                  70304|              70304|
|   mean|74928.19485870609| 573723.0996046612| 476.3754409421939|         37216.430748576764|          284234.31813326856|-7.43973465606391E-4|    0.15904091546121085|  0.503179139687885|
| stddev| 352536.

In [13]:
# 3.3 Outlier detection and treatment (p1/p99 winsorization).
# relativeError=0.001 so the p99 estimate does not collapse onto the maximum
# (with 0.01 the bounds equalled the max and the winsorization had no effect).
winsor_cols = ["volume", "quote_asset_volume", "number_of_trades",
               "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"]

bounds = {}
for c in winsor_cols:
    lo, hi = M_clean.approxQuantile(c, [0.01, 0.99], 0.001)
    bounds[c] = (lo, hi)
    print(f"{c:>30}: p1={lo:,.4f}  p99={hi:,.4f}")

M_prep = M_clean
for c, (lo, hi) in bounds.items():
    M_prep = M_prep.withColumn(
        c, when(col(c) < lo, lo).when(col(c) > hi, hi).otherwise(col(c))
    )

# return_pct: clip extreme returns (errors / flash crashes) to +/- 25%
M_prep = M_prep.withColumn(
    "return_pct",
    when(col("return_pct") > 25, 25.0).when(col("return_pct") < -25, -25.0).otherwise(col("return_pct")),
).cache()

print(f"\nPre-processed sample M: {M_prep.count():,} instances")
M_prep.select(num_cols).describe().show()

                        volume: p1=0.1036  p99=1,262,243.0000
            quote_asset_volume: p1=69.2718  p99=6,288,009.5000
              number_of_trades: p1=1.0000  p99=4,500.0000
   taker_buy_base_asset_volume: p1=0.0000  p99=645,099.0000
  taker_buy_quote_asset_volume: p1=0.0000  p99=3,221,012.2500

Pre-processed sample M: 70,304 instances
+-------+-------------------+------------------+-----------------+---------------------------+----------------------------+--------------------+-----------------------+-------------------+
|summary|             volume|quote_asset_volume| number_of_trades|taker_buy_base_asset_volume|taker_buy_quote_asset_volume|          return_pct|intrabar_volatility_pct|    taker_buy_ratio|
+-------+-------------------+------------------+-----------------+---------------------------+----------------------------+--------------------+-----------------------+-------------------+
|  count|              70304|             70304|            70304|                    

# 4. Preparación del conjunto de entrenamiento y prueba

**Técnica:** *split* aleatorio **estratificado por la variable objetivo**
(`market_direction`) usando `sampleBy()`. Como las clases están desbalanceadas
(neutral ≈ 5 %), un *split* puramente aleatorio podría dejar muy pocas instancias
neutrales en alguno de los conjuntos. El muestreo estratificado **garantiza que cada
clase conserve la misma proporción 80/20** en train y test, minimizando el sesgo.

**Porcentaje 80 / 20 — justificación:** es el estándar más usado en la práctica.
Con 80 % de los datos el modelo dispone de suficientes ejemplos por clase para
aprender patrones estables, mientras que el 20 % restante es un conjunto de prueba lo
bastante grande para estimar la generalización con baja varianza. Dado el tamaño de
*M* (decenas de miles de instancias), el 20 % sigue siendo un test representativo.


In [14]:
# 4.1 Stratified 80/20 split with a unique id to guarantee disjoint sets
from pyspark.sql.functions import monotonically_increasing_id

M_id = M_prep.withColumn("row_id", monotonically_increasing_id()).cache()

labels = [r["market_direction"] for r in M_id.select("market_direction").distinct().collect()]
train_fractions = {lbl: 0.8 for lbl in labels}

train = M_id.sampleBy("market_direction", fractions=train_fractions, seed=7).cache()
# test = M \ train  (anti-join by id => disjoint sets)
test = M_id.join(train.select("row_id"), on="row_id", how="left_anti").cache()

print(f"Train: {train.count():,}   Test: {test.count():,}")

Train: 56,229   Test: 14,075


In [15]:
# 4.2 Check: class proportions are preserved
print("=== market_direction distribution ===")
for name, dset in [("TRAIN", train), ("TEST", test)]:
    print(f"\n{name}")
    (dset.groupBy("market_direction")
         .agg(count("*").alias("n"))
         .orderBy("market_direction")
         .show())

=== market_direction distribution ===

TRAIN
+----------------+-----+
|market_direction|    n|
+----------------+-----+
|         bearish|26712|
|         bullish|26845|
|         neutral| 2672|
+----------------+-----+


TEST
+----------------+----+
|market_direction|   n|
+----------------+----+
|         bearish|6686|
|         bullish|6670|
|         neutral| 719|
+----------------+----+



# 5. Construcción de modelos

## 5.1 Aprendizaje supervisado — `RandomForestClassifier`

**Variable objetivo:** `market_direction` (bullish / bearish / neutral).

**Características (features):** deliberadamente **excluimos `open`, `high`, `low`,
`close` y `return_pct`**, porque `market_direction` se deriva del signo de
`(close − open)` y usarlas sería **fuga de información** (el modelo "haría trampa").
En su lugar planteamos un problema genuino: *¿se puede predecir la dirección de la
vela a partir de su microestructura de mercado?* Usamos:

- Numéricas: `volume`, `quote_asset_volume`, `number_of_trades`,
  `taker_buy_base_asset_volume`, `taker_buy_quote_asset_volume`, `taker_buy_ratio`,
  `intrabar_volatility_pct`, `hour`.
- Categóricas (codificadas con OneHot): `symbol`, `market_session`.

La señal clave es `taker_buy_ratio` (proporción de volumen agresor comprador): una
presión compradora alta tiende a empujar el precio al alza → *bullish*.

**Pipeline:** `StringIndexer` (etiqueta y categóricas) → `OneHotEncoder` →
`VectorAssembler` → `RandomForestClassifier`.


In [16]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

numeric_features = ["volume", "quote_asset_volume", "number_of_trades",
                    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume",
                    "taker_buy_ratio", "intrabar_volatility_pct", "hour"]

# Label -> numeric index
label_indexer = StringIndexer(inputCol="market_direction", outputCol="label", handleInvalid="keep")

# Categoricals -> index -> one-hot
cat_cols = ["symbol", "market_session"]
cat_indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
cat_encoder = OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols],
                            outputCols=[c + "_oh" for c in cat_cols])

assembler = VectorAssembler(
    inputCols=numeric_features + [c + "_oh" for c in cat_cols],
    outputCol="features",
)

rf = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=80, maxDepth=10, seed=42,
)

pipeline = Pipeline(stages=[label_indexer] + cat_indexers + [cat_encoder, assembler, rf])

print("Training RandomForest...")
rf_model = pipeline.fit(train)
print("Model trained.")

Training RandomForest...
Model trained.


In [17]:
# 5.1.1 Prediction and evaluation
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

pred = rf_model.transform(test)

metrics = {}
for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    ev = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=m)
    metrics[m] = ev.evaluate(pred)

print("=== Metrics on the test set ===")
for k, v in metrics.items():
    print(f"  {k:>18}: {v:.4f}")

=== Metrics on the test set ===
            accuracy: 0.6407
                  f1: 0.6386
   weightedPrecision: 0.6468
      weightedRecall: 0.6407


In [18]:
# 5.1.2 Confusion matrix
# Index -> original label mapping
label_meta = rf_model.stages[0].labels
print("Label order (index -> class):", dict(enumerate(label_meta)))

print("\n=== Confusion matrix (rows = actual, columns = predicted) ===")
pred.groupBy("label").pivot("prediction").count().orderBy("label").show()

Label order (index -> class): {0: 'bullish', 1: 'bearish', 2: 'neutral'}

=== Confusion matrix (rows = actual, columns = predicted) ===
+-----+----+----+---+
|label| 0.0| 1.0|2.0|
+-----+----+----+---+
|  0.0|4410|2232| 28|
|  1.0|2316|4350| 20|
|  2.0| 214| 247|258|
+-----+----+----+---+



In [19]:
# 5.1.3 Feature importances
rf_stage = rf_model.stages[-1]
importances = rf_stage.featureImportances

imp = importances.toArray()
print("=== Feature importances (Random Forest) ===")
# Numeric features occupy 1 position each at the start of the vector
for i, name in enumerate(numeric_features):
    print(f"  {name:>30}: {imp[i]:.4f}")
print("  (the remaining positions correspond to the one-hot categoricals)")

=== Feature importances (Random Forest) ===
                          volume: 0.0421
              quote_asset_volume: 0.0689
                number_of_trades: 0.0728
     taker_buy_base_asset_volume: 0.0472
    taker_buy_quote_asset_volume: 0.0505
                 taker_buy_ratio: 0.5102
         intrabar_volatility_pct: 0.1601
                            hour: 0.0280
  (the remaining positions correspond to the one-hot categoricals)


### Discusión — modelo supervisado

**Desempeño global.** Sobre el conjunto de prueba (14 075 instancias) el modelo
obtuvo **accuracy = 0.641** y **F1 = 0.639** (precision ponderada 0.647). El
*baseline* de predecir siempre la clase mayoritaria (*bullish*, ≈ 47 % del test)
daría ≈ 0.47, por lo que el modelo **supera al azar informado en ~17 puntos**: la
microestructura de mercado **sí contiene señal** sobre la dirección de la vela.

**Análisis por clase (matriz de confusión).** El *recall* por clase es:
- *bullish*: 4 410 / 6 670 ≈ **66 %**
- *bearish*: 4 350 / 6 686 ≈ **65 %**
- *neutral*: 258 / 719 ≈ **36 %**

Bullish y bearish se separan bien; el error dominante es confundir una con otra
(2 232 y 2 316 casos), algo esperable porque ambas dependen de un balance fino de
presión compradora/vendedora. La clase **neutral** es la más difícil por ser
minoritaria (≈ 5 %) y porque corresponde a velas planas (`close == open`) sin un
patrón de volumen claro.

**Variables más informativas.** La importancia confirma la hipótesis económica:
`taker_buy_ratio` domina con **0.51**, seguida de `intrabar_volatility_pct`
(**0.16**) y `number_of_trades` (**0.07**). Es decir, **la proporción de volumen
agresor comprador es el predictor más fuerte de la dirección**, coherente con la
teoría de microestructura: el flujo de órdenes agresoras mueve el precio. La hora del
día (`hour`, 0.03) aporta poco.

**Posibles mejoras.** Balancear la clase *neutral* (p. ej. con pesos o sobre-muestreo),
ajustar `numTrees`/`maxDepth` con `CrossValidator`, o comparar contra `GBTClassifier`.


## 5.2 Aprendizaje no supervisado — `KMeans`

**Objetivo:** descubrir **regímenes de mercado** (grupos de velas con comportamiento
similar) sin usar ninguna etiqueta.

**Características de agrupamiento:** `volume`, `number_of_trades`, `return_pct`,
`intrabar_volatility_pct`, `taker_buy_ratio`. K-Means se basa en distancias
euclidianas, por lo que es **imprescindible estandarizar** (media 0, desviación 1)
con `StandardScaler`; de lo contrario el `volume` (escala enorme) dominaría a las
demás variables.

**Selección de k:** probamos k = 2…6 y elegimos el de mayor **coeficiente de silueta**.


In [20]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

cluster_features = ["volume", "number_of_trades", "return_pct",
                    "intrabar_volatility_pct", "taker_buy_ratio"]

assembler_c = VectorAssembler(inputCols=cluster_features, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

# Prepare the scaled data once (we use the whole pre-processed M)
prep_pipeline = Pipeline(stages=[assembler_c, scaler])
prep_model = prep_pipeline.fit(M_prep)
data_scaled = prep_model.transform(M_prep).cache()
print("Scaled data ready:", data_scaled.count(), "instances")

Scaled data ready: 70304 instances


In [21]:
# 5.2.1 Choosing k via silhouette
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")

results = []
for k in range(2, 7):
    km = KMeans(featuresCol="features", predictionCol="prediction", k=k, seed=42)
    model_k = km.fit(data_scaled)
    pred_k = model_k.transform(data_scaled)
    sil = evaluator.evaluate(pred_k)
    wssse = model_k.summary.trainingCost
    results.append((k, sil, wssse))
    print(f"k={k}:  silhouette={sil:.4f}   WSSSE(inertia)={wssse:,.1f}")

best_k = max(results, key=lambda r: r[1])[0]
print(f"\nBest k by silhouette: k = {best_k}")

k=2:  silhouette=0.2610   WSSSE(inertia)=299,770.5
k=3:  silhouette=0.3392   WSSSE(inertia)=240,180.7
k=4:  silhouette=0.7078   WSSSE(inertia)=220,779.5
k=5:  silhouette=0.4309   WSSSE(inertia)=175,528.5
k=6:  silhouette=0.4221   WSSSE(inertia)=163,354.1

Best k by silhouette: k = 4


In [22]:
# 5.2.2 Final K-Means model
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=best_k, seed=42)
kmeans_model = kmeans.fit(data_scaled)

clustered = kmeans_model.transform(data_scaled).cache()

print(f"=== K-Means with k={best_k} ===")
print("Size of each cluster:")
clustered.groupBy("cluster").agg(count("*").alias("n")).orderBy("cluster").show()

=== K-Means with k=4 ===
Size of each cluster:
+-------+-----+
|cluster|    n|
+-------+-----+
|      0|62137|
|      1| 4341|
|      2| 2538|
|      3| 1288|
+-------+-----+



In [23]:
# 5.2.3 Interpretation: mean profile of each cluster (in original scale)
from pyspark.sql.functions import avg, round as sround

print("=== Profile of each cluster (means in original units) ===")
(clustered.groupBy("cluster")
    .agg(
        sround(avg("volume"), 2).alias("volume"),
        sround(avg("number_of_trades"), 1).alias("n_trades"),
        sround(avg("return_pct"), 4).alias("return_pct"),
        sround(avg("intrabar_volatility_pct"), 4).alias("volat_pct"),
        sround(avg("taker_buy_ratio"), 3).alias("buy_ratio"),
    )
    .orderBy("cluster")
    .show(truncate=False))

print("=== Symbol composition within each cluster ===")
clustered.groupBy("cluster").pivot("symbol").count().orderBy("cluster").show()

=== Profile of each cluster (means in original units) ===
+-------+---------+--------+----------+---------+---------+
|cluster|volume   |n_trades|return_pct|volat_pct|buy_ratio|
+-------+---------+--------+----------+---------+---------+
|0      |32322.74 |261.5   |-0.0042   |0.1247   |0.503    |
|1      |1124.86  |2699.1  |-0.0499   |0.2689   |0.485    |
|2      |861783.26|889.4   |-0.0887   |0.4947   |0.482    |
|3      |118457.5 |863.2   |0.5031    |0.7861   |0.605    |
+-------+---------+--------+----------+---------+---------+

=== Symbol composition within each cluster ===
+-------+-------+-------+-------+
|cluster|BTCUSDT|ETHUSDT|XRPUSDT|
+-------+-------+-------+-------+
|      0|  21008|  22857|  18272|
|      1|   3239|   1097|      5|
|      2|     12|     26|   2500|
|      3|    356|    620|    312|
+-------+-------+-------+-------+



### Discusión — modelo no supervisado

**Selección de k.** El coeficiente de silueta fue 0.26 (k=2), 0.34 (k=3), **0.71
(k=4)**, 0.43 (k=5) y 0.42 (k=6). El máximo claro en **k = 4** indica que cuatro
grupos capturan mejor la estructura. La inercia (WSSSE) baja de forma monótona con k,
por eso usamos la silueta como criterio principal y el codo solo como apoyo.

**Regímenes de mercado encontrados (k = 4).** Interpretando los centroides en
unidades originales:

| Cluster | n | volumen | n_trades | return % | volat % | buy_ratio | Interpretación |
|---|---|---|---|---|---|---|---|
| **0** | 62 137 | 32 k | 261 | ≈ 0 | 0.12 | 0.50 | **Mercado en calma**: baja actividad y volatilidad, dirección neutra. Es el régimen normal (88 % de las velas). |
| **1** | 4 341 | 1.1 k | 2 699 | −0.05 | 0.27 | 0.49 | **Muchas operaciones pequeñas**: alto nº de trades con volumen bajo (alta fragmentación). |
| **2** | 2 538 | 862 k | 889 | −0.09 | 0.49 | 0.48 | **Picos de liquidez**: volumen extremo y volatilidad alta. |
| **3** | 1 288 | 118 k | 863 | **+0.50** | **0.79** | **0.61** | **Rallies alcistas**: retorno positivo, fuerte presión compradora y la mayor volatilidad. |

**Composición por símbolo (coherencia con la Evidencia 1).** El régimen de **picos
de liquidez (cluster 2) es casi 100 % XRP** (2 500 de 2 538), confirmando que XRP es
el activo más especulativo y de mayor volumen en unidades base (por su bajo precio).
El de **alta fragmentación de trades (cluster 1) está dominado por BTC** (3 239 de
4 341), consistente con su microestructura de muchas órdenes pequeñas. El régimen de
calma (cluster 0) reparte los tres pares de forma equilibrada.

**Lectura final.** K-Means logró separar **estados de mercado con sentido económico**
(calma, fragmentación, picos de liquidez y rallies) sin usar ninguna etiqueta, y los
grupos resultan consistentes con la caracterización por activo de la Evidencia 1.

*(Alternativa: `GaussianMixture` modela clusters elípticos con probabilidad de
pertenencia, útil cuando los regímenes se solapan en las fronteras.)*


## 6. Conclusiones

- Se aplicó con éxito el flujo completo de aprendizaje en PySpark sobre datos de
  Binance: **selección estratificada** de la muestra *M*, **pre-procesamiento**
  (nulos, outliers, tipos), **split estratificado 80/20** y entrenamiento de un
  modelo **supervisado** (RandomForest) y uno **no supervisado** (K-Means).
- El diseño es **continuo con la Evidencia 1**: mismas variables de caracterización,
  mismo particionamiento A×B×C y misma técnica de muestreo (`sampleBy`).
- Se evitó la **fuga de información** en el modelo supervisado y se **estandarizaron**
  las variables para el clustering, dos buenas prácticas que dan validez a los
  resultados obtenidos.
